In [ ]:
# PyTorch 与神经网络模块
import torch
from torch import nn
# 课程仓库里的硬件档案与仿真环境
import hardware
from env import DuckEnv, SKILLS

In [ ]:
# 走路技能、10 月 24 日这一版硬件；2048 只鸭子并行，固定随机种子
env = DuckEnv(SKILLS["walk"], hardware.load("2026-11-07-hd1910"), num_envs=2048, device="cuda", seed=1)
obs, critic_obs = env.reset()  # 拿到第一批观测：2048 × 61

In [ ]:
class Policy(nn.Module):
    def __init__(self):
        super().__init__()
        # 多层感知机：61 个观测 → 512 → 256 → 128 → 14 个动作的均值
        self.actor = nn.Sequential(nn.Linear(61, 512), nn.ELU(), nn.Linear(512, 256), nn.ELU(),
                                   nn.Linear(256, 128), nn.ELU(), nn.Linear(128, 14))
        # 14 个标准差，单独训练；初值 1，开始时探索的幅度大
        self.std = nn.Parameter(torch.ones(14))
        # 观测的均值与方差，用来做归一化（不参与梯度）
        self.register_buffer("obs_mean", torch.zeros(61))
        self.register_buffer("obs_var", torch.ones(61))
        self.register_buffer("count", torch.zeros(()))

    def update_normalizer(self, obs):
        # 每来一批观测，统计量往这一批靠一点；见过的越多，每次挪得越少
        self.count += len(obs)
        w = (len(obs) / self.count).item()
        self.obs_mean.lerp_(obs.mean(0), w)
        self.obs_var.lerp_(obs.var(0), w)

    def dist(self, obs):
        # 观测先归一化，网络算均值，和标准差一起组成 14 维高斯分布
        x = (obs - self.obs_mean) / (self.obs_var.sqrt() + 1e-2)
        return torch.distributions.Normal(self.actor(x), self.std)


In [ ]:
policy = Policy().to("cuda")
action = policy.dist(obs).sample()  # 从分布里采一个动作
action.shape  # 2048 只鸭子 × 14 个关节

In [ ]:
# 第一步，采样：存下观测、动作、奖励、是否结束，每只鸭子走 24 步
obs_buf, act_buf, rew_buf, done_buf = [], [], [], []
for t in range(24):
    with torch.no_grad():  # 采样不需要梯度
        action = policy.dist(obs).sample()
    next_obs, _, reward, done, _ = env.step(action)  # 送进环境，走一步
    obs_buf.append(obs); act_buf.append(action)  # 存下这一步
    rew_buf.append(reward); done_buf.append(done.float())
    policy.update_normalizer(next_obs)  # 更新归一化的统计量
    obs = next_obs
len(rew_buf), rew_buf[0].shape  # 24 步，每步 2048 条

In [ ]:
# 第二步，算回报：从后往前，u_t = r_t + 0.99 × u_{t+1}，回合结束处乘 0
rewards, dones = torch.stack(rew_buf), torch.stack(done_buf)
returns = torch.zeros_like(rewards)
u = torch.zeros_like(rewards[0])
for t in reversed(range(24)):
    u = rewards[t] + 0.99 * u * (1 - dones[t])
    returns[t] = u
# 减去这一批的平均（基线），再除以标准差，作为每条数据的权重
adv = (returns - returns.mean()) / (returns.std() + 1e-8)
returns.mean()

In [ ]:
# 第三步，算损失；学习率不能大，大了一步迈过头
optimizer = torch.optim.Adam(policy.parameters(), lr=3e-4)
dist = policy.dist(torch.cat(obs_buf))  # 24 步的观测拼起来，重新算分布
log_prob = dist.log_prob(torch.cat(act_buf)).sum(-1)  # 14 个关节的 log 概率相加
entropy = dist.entropy().sum(-1).mean()  # 熵：分布有多宽
# 目标要变大，优化器做最小化，所以加负号；减去熵，鼓励探索
loss = -(log_prob * adv.flatten()).mean() - 0.01 * entropy
optimizer.zero_grad()  # 清空梯度
loss.backward()  # 反向传播
optimizer.step()  # 更新一次参数
loss

In [ ]:
# Lightning 把训练循环收起来，我们只写数据、损失和优化器
import lightning as L
from torch.utils.data import DataLoader, IterableDataset

In [ ]:
# 数据：采样和算回报，搬进一个数据集
class Rollout(IterableDataset):
    def __init__(self, env, policy):
        super().__init__()
        self.env, self.policy = env, policy
        self.batch = None

    def __iter__(self):
        # Lightning 每轮会读两次数据；一个数据集只采一次样，免得白采
        if self.batch is None:
            self.batch = self.sample()
        yield self.batch

    def sample(self):
        obs, _ = self.env.get_observations()  # 接着环境当前的状态往下采
        # 采样：和第一步那一格一样，再数一数摔倒
        obs_buf, act_buf, rew_buf, done_buf = [], [], [], []
        falls = 0
        for t in range(24):
            with torch.no_grad():
                action = self.policy.dist(obs).sample()
            next_obs, _, reward, done, info = self.env.step(action)
            falls += info["terminated"].sum().item()  # 数摔倒次数
            obs_buf.append(obs); act_buf.append(action)
            rew_buf.append(reward); done_buf.append(done.float())
            self.policy.update_normalizer(next_obs)
            obs = next_obs
        # 算回报和权重：和第二步那一格一样
        rewards, dones = torch.stack(rew_buf), torch.stack(done_buf)
        returns = torch.zeros_like(rewards)
        u = torch.zeros_like(rewards[0])
        for t in reversed(range(24)):
            u = rewards[t] + 0.99 * u * (1 - dones[t])
            returns[t] = u
        adv = (returns - returns.mean()) / (returns.std() + 1e-8)
        # 打包成一批，交给训练
        return {"obs": torch.cat(obs_buf), "actions": torch.cat(act_buf), "adv": adv.flatten(),
                "reward": rewards.mean(), "falls": falls}

# 数据模块：每一轮训练给出一个新的数据集，也就是重新采样
class DuckData(L.LightningDataModule):
    def __init__(self, env, policy):
        super().__init__()
        self.env, self.policy = env, policy

    def train_dataloader(self):
        return DataLoader(Rollout(self.env, self.policy), batch_size=None)

# 模型：损失和刚才那一格一样
class Reinforce(L.LightningModule):
    def __init__(self, policy):
        super().__init__()
        self.policy = policy
        self.history = []

    def training_step(self, batch, batch_idx):
        dist = self.policy.dist(batch["obs"])
        log_prob = dist.log_prob(batch["actions"]).sum(-1)
        entropy = dist.entropy().sum(-1).mean()
        loss = -(log_prob * batch["adv"]).mean() - 0.01 * entropy
        self.history.append((batch["reward"].item(), batch["falls"]))  # 记下这一轮的奖励和摔倒
        self.log("reward", batch["reward"], prog_bar=True)
        return loss

    # 优化器：清空梯度、反向传播、走一步，都由 Lightning 来做
    def configure_optimizers(self):
        return torch.optim.Adam(self.policy.parameters(), lr=3e-4)

In [ ]:
# 评测用的环境：16 只鸭子，打开渲染
test = DuckEnv(SKILLS["walk"], hardware.load("2026-11-07-hd1910"), num_envs=16, device="cuda", seed=1, render=True)

# 评测：每只鸭子跑 400 步，数摔倒次数、录下画面
def evaluate(policy):
    obs, _ = test.reset()
    falls, frames = 0, []
    for t in range(400):
        with torch.no_grad():
            obs, _, reward, done, info = test.step(policy.dist(obs).mean)  # 评测不采样，直接用均值
        falls += info["terminated"].sum().item()
        frames.append(test.render())
    return falls, frames

policy = Policy().to("cuda")
evaluate(policy)[0]  # 没训练的策略先测一次，当对照

In [ ]:
# 训 1000 轮；每轮重新采样；梯度范数裁到 1，免得一步走太大
model = Reinforce(policy)
trainer = L.Trainer(max_epochs=1000, reload_dataloaders_every_n_epochs=1,
                    gradient_clip_val=1.0, logger=False, enable_checkpointing=False)
trainer.fit(model, DuckData(env, policy))

In [ ]:
# 画两条曲线：每一轮的平均奖励、每一轮的摔倒次数
import matplotlib.pyplot as plt
reward, falls = zip(*model.history)
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(reward); ax[0].set_title("reward per step")
ax[1].plot(falls); ax[1].set_title("falls per iteration");

In [ ]:
policy.to("cuda")  # Lightning 训完把模型挪回了 CPU，先放回显卡
falls, frames = evaluate(policy)
falls

In [ ]:
# 把评测画面放出来看
import mediapy
mediapy.show_video(frames, fps=50, width=640)

In [ ]:
# 存下策略和曲线，下节课对比用
torch.save({"policy": policy.state_dict(), "history": model.history}, "reinforce.pt")